# Tutorial 06: Boosted Decision Trees (BDTs) em FPGA com Conifer

Neste tutorial, exploramos a aceleração de **Árvores de Decisão Aumentadas por Gradiente (BDTs)** em FPGAs utilizando o compilador de código aberto **Conifer**.

### Por que BDTs em FPGA?
- **Zero Multiplicadores DSP**: Ao contrário das Redes Neurais que dependem fortemente de multiplicações matriciais densas, as Árvores de Decisão consistem puramente em **comparações de limiar** ($x_i \le \theta$). No silício da FPGA, essas comparações são mapeadas com extrema densidade em blocos de tabelas de busca (**LUTs**);
- **Latência Ultrabaixa (Nanossegundos)**: Todas as árvores do ensemble podem ser avaliadas em paralelo em um único ciclo de clock ou numa árvore rasa de multiplexadores;
- **Padrão em Gatilhos de Partículas (LHC)**: Pela sua velocidade e interpretabilidade direta, BDTs são amplamente empregados nos experimentos CMS e ATLAS.

## 1. Importação de Bibliotecas e Configuração dos Cabeçalhos

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_curve, auc, accuracy_score
import joblib

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

import conifer
import hls4ml
import nlohmann_json

# Configurar links para cabeçalhos de C++ necessários pelo Conifer
c_path = conifer.__path__[0]
h_path = os.path.dirname(hls4ml.__file__)
n_path = os.path.dirname(nlohmann_json.__file__)
os.makedirs(f"{c_path}/external/json", exist_ok=True)
if not os.path.exists(f"{c_path}/external/json/include"):
    try: os.symlink(f"{n_path}/include", f"{c_path}/external/json/include")
    except OSError: pass
os.makedirs(f"{c_path}/external/ap_types", exist_ok=True)
if not os.path.exists(f"{c_path}/external/ap_types/include"):
    try: os.symlink(f"{h_path}/templates/vivado/ap_types", f"{c_path}/external/ap_types/include")
    except OSError: pass

print(f"Versão Conifer: {conifer.__version__}")

## 2. Carregando os Dados de Jatos do CERN LHC
Definimos o problema de classificação binária: identificar jatos de **Quarks Top (`t`)** contra o fundo de jatos leves e glúons.

In [ ]:
data_dir = os.path.join(ROOT_DIR, "data", "jet-tagging")
X_train = np.load(os.path.join(data_dir, "X_train.npy"))
y_train = np.load(os.path.join(data_dir, "y_train.npy"))
X_test = np.load(os.path.join(data_dir, "X_test.npy"))
y_test = np.load(os.path.join(data_dir, "y_test.npy"))
classes = list(np.load(os.path.join(data_dir, "classes.npy"), allow_pickle=True))

idx_t = classes.index("t")
y_train_bin = (y_train[:, idx_t] == 1).astype(int)
y_test_bin = (y_test[:, idx_t] == 1).astype(int)

n_train = 50000
X_train_sub = X_train[:n_train]
y_train_sub = y_train_bin[:n_train]

eval_n = 10000
X_eval = X_test[:eval_n]
y_eval = y_test_bin[:eval_n]

print(f"Treino: {X_train_sub.shape} | Teste Avaliação: {X_eval.shape}")

## 3. Treinamento da BDT no scikit-learn
Treinamos um `GradientBoostingClassifier` com 30 estimadores (árvores) e profundidade máxima 3.

In [ ]:
model_path = os.path.join(ROOT_DIR, "models", "sklearn_bdt_model.joblib")
if os.path.exists(model_path):
    print(f"Carregando modelo salvo de {model_path}...")
    clf = joblib.load(model_path)
else:
    print("Treinando modelo BDT...")
    clf = GradientBoostingClassifier(n_estimators=30, max_depth=3, learning_rate=0.2, random_state=42)
    clf.fit(X_train_sub, y_train_sub)
    joblib.dump(clf, model_path)

scores_sklearn = clf.decision_function(X_eval)
preds_sklearn = (scores_sklearn > 0).astype(int)
acc_sklearn = accuracy_score(y_eval, preds_sklearn)
fpr_sk, tpr_sk, _ = roc_curve(y_eval, scores_sklearn)
auc_sk = auc(fpr_sk, tpr_sk)

print(f"Acurácia scikit-learn (Float64): {acc_sklearn * 100:.2f}%")
print(f"AUC scikit-learn (Float64)     : {auc_sk:.4f}")

## 4. Configuração e Conversão de Hardware com Conifer
Configuramos a precisão de ponto fixo `ap_fixed<18,8>` e o backend nativo `cpp`.

In [ ]:
output_dir = "conifer_prj_bdt"
cfg = conifer.backends.cpp.auto_config()
cfg["OutputDir"] = output_dir
cfg["Precision"] = "ap_fixed<18,8>"

model_conifer = conifer.converters.convert_from_sklearn(clf, cfg)
print("Modelo BDT convertido com sucesso para Conifer!")

## 5. Compilação e Simulação Nativa com `g++`

In [ ]:
t0 = time.time()
model_conifer.compile()
print(f"Compilação g++ finalizada em {time.time() - t0:.2f}s.")

t1 = time.time()
scores_conifer = model_conifer.decision_function(X_eval).ravel()
print(f"Simulação de hardware em {eval_n} eventos concluída em {time.time() - t1:.2f}s.")

preds_conifer = (scores_conifer > 0).astype(int)
acc_conifer = accuracy_score(y_eval, preds_conifer)
fpr_cn, tpr_cn, _ = roc_curve(y_eval, scores_conifer)
auc_cn = auc(fpr_cn, tpr_cn)

print(f"Acurácia scikit-learn (Float64) : {acc_sklearn * 100:.2f}%")
print(f"Acurácia Conifer (ap_fixed<18,8>): {acc_conifer * 100:.2f}%")
print(f"AUC scikit-learn (Float64)      : {auc_sk:.4f}")
print(f"AUC Conifer (ap_fixed<18,8>)    : {auc_cn:.4f}")
print(f"Diferença de Acurácia           : {abs(acc_sklearn - acc_conifer) * 100:.3f}%")
print(f"Diferença Absoluta de AUC       : {abs(auc_sk - auc_cn):.5f}")

## 6. Curvas ROC Comparativas (scikit-learn vs Conifer Hardware)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr_sk, tpr_sk, "b--", lw=1.8, label=f"scikit-learn Float64 (AUC = {auc_sk:.3f})")
plt.plot(fpr_cn, tpr_cn, "r-", lw=1.8, label=f"Conifer ap_fixed<18,8> (AUC = {auc_cn:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Baseline Aleatório (AUC = 0.500)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Taxa de Falso Positivo (FPR)", fontsize=11)
plt.ylabel("Taxa de Verdadeiro Positivo (TPR)", fontsize=11)
plt.title("Classificação de Top Quarks com BDT em FPGA (Conifer vs scikit-learn)", fontsize=12, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()